In [1]:
pip install torch nltk numpy scikit-learn sacrebleu


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 6.4 MB/s eta 0:00:00


In [2]:
import nltk
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from nltk.tokenize import word_tokenize
from nltk.util import ngrams
from collections import Counter, defaultdict
from sacrebleu import corpus_bleu

nltk.download('punkt')


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

In [4]:
import nltk
nltk.download('punkt_tab')

corpus = """
Viral fever is a common illness caused by viral infections.
It usually spreads through air or contact with infected people.
Common symptoms include high temperature, headache, sore throat, and muscle pain.
Rest, hydration, and proper medication help in recovery.
In severe cases, patients may experience fatigue and weakness for several days.
"""

tokens = word_tokenize(corpus.lower())
print("Total Tokens:", len(tokens))
print("First 20 Tokens:", tokens[:20])

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


Total Tokens: 62
First 20 Tokens: ['viral', 'fever', 'is', 'a', 'common', 'illness', 'caused', 'by', 'viral', 'infections', '.', 'it', 'usually', 'spreads', 'through', 'air', 'or', 'contact', 'with', 'infected']


In [5]:
def train_ngram_model(tokens, n):
    model = defaultdict(Counter)
    for gram in ngrams(tokens, n, pad_left=True, pad_right=True):
        context = gram[:-1]
        word = gram[-1]
        model[context][word] += 1
    return model

bigram_model = train_ngram_model(tokens, 2)
trigram_model = train_ngram_model(tokens, 3)

def predict_ngram(context, model):
    context = tuple(context)
    if context in model:
        return model[context].most_common(1)[0][0]
    return None


In [6]:
def calculate_ngram_perplexity(model, tokens, n):
    N = 0
    log_prob = 0

    for gram in ngrams(tokens, n, pad_left=True, pad_right=True):
        context = gram[:-1]
        word = gram[-1]

        freq = model[context][word] + 1  # Laplace smoothing
        total = sum(model[context].values()) + len(model)

        prob = freq / total
        log_prob += np.log(prob)
        N += 1

    return np.exp(-log_prob / N)

print("Bigram Perplexity:", calculate_ngram_perplexity(bigram_model, tokens, 2))
print("Trigram Perplexity:", calculate_ngram_perplexity(trigram_model, tokens, 3))


Bigram Perplexity: 25.1521332319644
Trigram Perplexity: 32.01550794307709


In [7]:
vocab = list(set(tokens))
word2idx = {w:i for i,w in enumerate(vocab)}
idx2word = {i:w for w,i in word2idx.items()}

encoded = [word2idx[w] for w in tokens]

def make_sequences(data, seq_len=5):
    x, y = [], []
    for i in range(len(data) - seq_len):
        x.append(data[i:i+seq_len])
        y.append(data[i+1:i+seq_len+1])
    return torch.tensor(x), torch.tensor(y)

X, Y = make_sequences(encoded)


In [8]:
class RNNModel(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size):
        super(RNNModel, self).__init__()
        self.embed = nn.Embedding(vocab_size, embed_size)
        self.rnn = nn.RNN(embed_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x):
        x = self.embed(x)
        out, _ = self.rnn(x)
        out = self.fc(out)
        return out

vocab_size = len(vocab)
model = RNNModel(vocab_size, 50, 100)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)


In [9]:
epochs = 50

for epoch in range(epochs):
    optimizer.zero_grad()
    output = model(X)
    loss = criterion(output.view(-1, vocab_size), Y.view(-1))
    loss.backward()
    optimizer.step()

    if (epoch+1) % 10 == 0:
        print(f"Epoch {epoch+1}/{epochs}, Loss: {loss.item():.4f}")


Epoch 10/50, Loss: 0.2599
Epoch 20/50, Loss: 0.0910
Epoch 30/50, Loss: 0.0849
Epoch 40/50, Loss: 0.0842
Epoch 50/50, Loss: 0.0839


In [10]:
def calculate_rnn_perplexity(loss):
    return torch.exp(loss).item()

print("RNN Perplexity:", calculate_rnn_perplexity(loss))


RNN Perplexity: 1.0875197649002075


In [11]:
def generate_ngram(model, seed_words, n, length=10):
    output = seed_words.copy()
    for _ in range(length):
        context = tuple(output[-(n-1):])
        next_word = model[context].most_common(1)[0][0] if context in model else None
        if next_word is None:
            break
        output.append(next_word)
    return output

ngram_generated = generate_ngram(bigram_model, ["viral"], 2)
print("Generated (N-gram):", " ".join(ngram_generated))


Generated (N-gram): viral fever is a common illness caused by viral fever is


In [12]:
def generate_rnn_text(model, start_word, num_words=10):
    model.eval()
    current = torch.tensor([[word2idx[start_word]]])
    result = [start_word]

    for _ in range(num_words):
        with torch.no_grad():
            output = model(current)
            predicted = output[:,-1,:].argmax(dim=1).item()
            result.append(idx2word[predicted])
            current = torch.tensor([[predicted]])

    return result

rnn_generated = generate_rnn_text(model, "viral")
print("Generated (RNN):", " ".join(rnn_generated))


Generated (RNN): viral infections . common illness caused by viral infections . common


In [13]:
reference = [" ".join(tokens)]

# N-gram BLEU Score
ngram_bleu = corpus_bleu([" ".join(ngram_generated)], [reference])
print("BLEU Score (N-gram):", ngram_bleu.score)

# RNN BLEU Score
rnn_bleu = corpus_bleu([" ".join(rnn_generated)], [reference])
print("BLEU Score (RNN):", rnn_bleu.score)


BLEU Score (N-gram): 0.7619333934757373
BLEU Score (RNN): 0.6284975226307259
